In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Shashwat-19/Object-Detection.git

In [ ]:
!ls

In [ ]:
%cd Object-Detection

In [ ]:
!ls

In [ ]:
import os
os.listdir("/content/drive/MyDrive/MOT17-02-FRCNN/img1")

In [ ]:
# Install Ultralytics if not already installed
!pip install ultralytics

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO
from IPython.display import Video

In [ ]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not enabled")

In [ ]:
# Load YOLOv8 medium model

model = YOLO("yolov8m.pt")

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/MOT17-02-FRCNN/img1"

results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    save=True,
    show=False,
    device=0,
    conf=0.3,
    persist=True
)

print("Tracking completed")

In [ ]:
os.listdir("runs/detect")

In [ ]:
TRACK_DIR = "runs/detect/track"
files = os.listdir(TRACK_DIR)
print(files[:20])

In [ ]:
# Display Tracked Output Video
from IPython.display import Video
import os

# The tracking results were saved as .jpg images in TRACK_DIR.
# The notebook is designed to convert these .jpg frames into 'tracked_output.mp4' in Cell 14.
# This cell now attempts to display the 'tracked_output.mp4' assuming it has been generated.
# If 'tracked_output.mp4' does not exist yet, please run Cell 14 first.

# Check if 'tracked_output.mp4' exists before trying to display it
output_video_path = "tracked_output.mp4"

if os.path.exists(output_video_path):
    print(f"Displaying {output_video_path}")
    Video(output_video_path, embed=True)
else:
    print(f"No .mp4 video found in TRACK_DIR, and '{output_video_path}' does not exist. Please run Cell 14 (Convert JPG Frames to MP4) to generate the video first.")

In [ ]:
# Convert JPG Frames to MP4
image_folder = TRACK_DIR
video_name = "tracked_output.mp4"

images = sorted(
    [img for img in os.listdir(image_folder) if img.endswith(".jpg")]
)

frame = cv2.imread(os.path.join(image_folder, images[0]))
height, width, layers = frame.shape

video = cv2.VideoWriter(
    video_name,
    cv2.VideoWriter_fourcc(*'mp4v'),
    30,
    (width, height)
)

for image in images:

    img_path = os.path.join(image_folder, image)

    frame = cv2.imread(img_path)

    video.write(frame)

video.release()

print("Video generated successfully")

In [ ]:
Video("tracked_output.mp4", embed=True)

In [ ]:
# Analyze tracking IDs

results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    show=False,
    save=False,
    device=0,
    stream=True
)

for frame_id, r in enumerate(results):

    if r.boxes.id is not None:

        ids = r.boxes.id.cpu().numpy()

        print(f"Frame {frame_id} -> IDs: {ids}")

In [ ]:
trajectory_history = {}

# Re-run tracking to get full results, ensuring persist=True for consistent IDs
results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    stream=True,
    persist=True,
    device=0
)

# Collect trajectory points for all objects across all frames
for r in results:
    if r.boxes.id is not None:
        boxes = r.boxes.xywh.cpu().numpy()
        ids = r.boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):
            x, y, w, h = box
            center = (int(x), int(y)) # Get center point

            if track_id not in trajectory_history:
                trajectory_history[track_id] = []
            trajectory_history[track_id].append(center)

# No longer saving a single image frame here, as the focus is on collecting full trajectories.
print("Full trajectory history collected for all frames.")

In [ ]:
heatmap = np.zeros((1080, 1920), dtype=np.float32)

results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    stream=True,
    device=0
)

for r in results:

    if r.boxes.xyxy is not None:

        boxes = r.boxes.xyxy.cpu().numpy()

        for box in boxes:

            x1, y1, x2, y2 = map(int, box)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            if cy < heatmap.shape[0] and cx < heatmap.shape[1]:
                heatmap[cy, cx] += 1

plt.figure(figsize=(12, 8))
plt.imshow(heatmap, cmap='hot')
plt.title("Movement Heatmap")
plt.colorbar()
plt.show()

In [ ]:
# Calculate horizontal and vertical movement distributions
horizontal_distribution = np.sum(heatmap, axis=0) # Summing activity vertically for each x-coordinate
vertical_distribution = np.sum(heatmap, axis=1)   # Summing activity horizontally for each y-coordinate

# Plot horizontal distribution
plt.figure(figsize=(12, 4))
plt.plot(horizontal_distribution)
plt.title("Horizontal Movement Distribution (Total Activity per X-coordinate)")
plt.xlabel("X-coordinate (pixels)")
plt.ylabel("Total Activity")
plt.grid(True)
plt.show()

# Plot vertical distribution
plt.figure(figsize=(12, 4))
plt.plot(vertical_distribution)
plt.title("Vertical Movement Distribution (Total Activity per Y-coordinate)")
plt.xlabel("Y-coordinate (pixels)")
plt.ylabel("Total Activity")
plt.grid(True)
plt.show()

In [ ]:
import time

start_time = time.time()

results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    stream=True,
    device=0
)

frame_count = 0

for _ in results:
    frame_count += 1

end_time = time.time()

fps = frame_count / (end_time - start_time)

print("Frames Processed:", frame_count)
print(f"FPS: {fps:.2f}")

## Quantitative Tracking Metrics

In [ ]:
# Number of unique objects tracked
unique_objects = len(trajectory_history)
print(f"Total Unique Objects Tracked: {unique_objects}")

# Calculate average trajectory length
trajectory_lengths = [len(points) for points in trajectory_history.values()]
if trajectory_lengths:
    average_length = sum(trajectory_lengths) / len(trajectory_lengths)
    print(f"Average Trajectory Length (frames): {average_length:.2f}")
else:
    print("No trajectories found.")

# Find the maximum number of objects tracked simultaneously
# This would require re-running tracking and counting objects per frame,
# or modifying the trajectory collection to store objects per frame.

In [ ]:
import os

output_video_path = "tracked_output.mp4"
destination_folder = "output"

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

destination_path = os.path.join(destination_folder, output_video_path)

# Move the file
if os.path.exists(output_video_path):
    os.rename(output_video_path, destination_path)
    print(f"Moved '{output_video_path}' to '{destination_path}'")
else:
    print(f"'{output_video_path}' not found in the current directory.")

## Individual Object Trajectory Graphs

In [ ]:
plt.figure(figsize=(15, 10))
plt.title("2D Trajectories of Individual Tracked Objects")
plt.xlabel("X-coordinate")
plt.ylabel("Y-coordinate")
plt.grid(True)

# Limit the number of trajectories to display for clarity
max_trajectories_to_show = 10

for i, (track_id, trajectory) in enumerate(trajectory_history.items()):
    if i >= max_trajectories_to_show: # Only plot the first N trajectories
        break

    # Unpack x and y coordinates
    x_coords = [p[0] for p in trajectory]
    y_coords = [p[1] for p in trajectory]

    plt.plot(x_coords, y_coords, label=f"Object {track_id}", linewidth=1.5)

# Add a legend if displaying multiple trajectories
if len(trajectory_history) > 1:
    plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# Invert y-axis to match image coordinates (origin top-left)
plt.gca().invert_yaxis()
plt.show()

print(f"Displaying trajectories for the first {min(len(trajectory_history), max_trajectories_to_show)} objects.")
print("You can modify `max_trajectories_to_show` in the code to visualize more or fewer trajectories.")